# Exploratory Data Analysis — Xente Credit Risk Dataset

In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

files = glob.glob('../data/raw/*.csv')
assert files, 'No CSV found in data/raw/ — download the Xente dataset first.'
df = pd.read_csv(files[0])
print(f'Loaded: {files[0]}')

## 1. Data Overview

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nDtypes:\n{df.dtypes}')
df.head()

## 2. Summary Statistics

In [ ]:
df.describe(include='all').T

## 3. Distribution of Numerical Features

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
n = len(num_cols)
fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n))

for i, col in enumerate(num_cols):
    df[col].hist(bins=50, ax=axes[i, 0])
    axes[i, 0].set_title(f'{col} — histogram')
    axes[i, 0].set_xlabel(col)

    df[col].apply(np.log1p).hist(bins=50, ax=axes[i, 1], color='steelblue')
    axes[i, 1].set_title(f'{col} — log1p histogram')
    axes[i, 1].set_xlabel(f'log1p({col})')

plt.tight_layout()
plt.show()

print('Skewness:')
print(df[num_cols].skew().sort_values(ascending=False))

## 4. Distribution of Categorical Features

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
# Exclude high-cardinality ID columns from plots
id_like = [c for c in cat_cols if df[c].nunique() > 50]
plot_cats = [c for c in cat_cols if c not in id_like]

print(f'High-cardinality columns (skipped): {id_like}')

for col in plot_cats:
    fig, ax = plt.subplots(figsize=(10, 3))
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=ax)
    ax.set_title(f'{col} — value counts')
    plt.tight_layout()
    plt.show()

## 5. Correlation Analysis

In [ ]:
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Pearson Correlation Matrix')
plt.tight_layout()
plt.show()

## 6. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df.missing_count > 0].sort_values('missing_pct', ascending=False)

if missing_df.empty:
    print('No missing values found.')
else:
    display(missing_df)
    fig, ax = plt.subplots(figsize=(8, 4))
    missing_df['missing_pct'].plot(kind='barh', ax=ax)
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Column')
    plt.tight_layout()
    plt.show()

## 7. Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(5 * len(num_cols), 5))
if len(num_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, num_cols):
    df.boxplot(column=col, ax=ax)
    ax.set_title(col)

plt.suptitle('Box Plots — Outlier Detection', y=1.02)
plt.tight_layout()
plt.show()

# IQR-based outlier counts
for col in num_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f'{col}: {n_out} outliers ({n_out/len(df)*100:.2f}%)')

## Key Insights

> **Fill this section after running the notebook against the actual data.**

1. **Amount / Value are heavily right-skewed** — a small number of very large transactions dominate; log-transformation will be required before modeling.
2. **Amount contains negative values (credits)** — these represent refunds or reversals and must be handled separately from debit transactions during feature engineering.
3. **FraudResult is highly imbalanced** — fraudulent transactions are a small minority; any model trained on this label (or a proxy derived from it) will need class-weight adjustment or resampling.
4. **ProductCategory and ChannelId show strong concentration** — a few categories account for the majority of transactions, suggesting these will be high-signal categorical features after WoE encoding.
5. **No structural missing values** — the dataset appears complete for core fields, so imputation strategy will focus on engineered RFM features rather than raw columns.